# GroupNorm & InstanceNorm

Wiki reference for [GroupNorm & InstanceNorm](https://ml-viz-ruby.vercel.app/wiki/groupnorm-and-instancenorm).

**The idea in one sentence.** The normalization family differs only in *which axes* it averages
over: BatchNorm pools across the batch (fragile for small batches), while **GroupNorm** pools
over channel groups within one sample — and it interpolates the whole family, equalling
**LayerNorm** at $G=1$ and **InstanceNorm** at $G=C$.

We implement all four norms from scratch, **validate the GroupNorm interpolation and
InstanceNorm's affine-invariance**, then cover the gotchas.

> **To save your work:** click **Copy to Drive**, or File → Save a copy in Drive.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.style.use('dark_background')
rng = np.random.default_rng(0)

## The four normalization variants, one function each

All four share the same normalize-then-rescale formula; only the axes that get
averaged over (the reduction set $S$) differ. `gamma`/`beta` are per-channel,
shape `(1, C, 1, 1)`, broadcast over the rest.

In [ ]:
def batch_norm(x, gamma, beta, eps=1e-5):
    """Reduce over (N, H, W) -> one stat per channel."""
    mu  = x.mean(axis=(0, 2, 3), keepdims=True)
    var = x.var(axis=(0, 2, 3), keepdims=True)
    x_hat = (x - mu) / np.sqrt(var + eps)
    return gamma * x_hat + beta


def layer_norm(x, gamma, beta, eps=1e-5):
    """Reduce over (C, H, W) -> one stat per sample."""
    mu  = x.mean(axis=(1, 2, 3), keepdims=True)
    var = x.var(axis=(1, 2, 3), keepdims=True)
    x_hat = (x - mu) / np.sqrt(var + eps)
    return gamma * x_hat + beta


def group_norm(x, G, gamma, beta, eps=1e-5):
    """Reduce over (C/G, H, W) -> one stat per (sample, group)."""
    N, C, H, W = x.shape
    assert C % G == 0, "channels must divide evenly into groups"
    x_g = x.reshape(N, G, C // G, H, W)
    mu  = x_g.mean(axis=(2, 3, 4), keepdims=True)
    var = x_g.var(axis=(2, 3, 4), keepdims=True)
    x_hat = (x_g - mu) / np.sqrt(var + eps)
    x_hat = x_hat.reshape(N, C, H, W)
    return gamma * x_hat + beta


def instance_norm(x, gamma, beta, eps=1e-5):
    """Reduce over (H, W) only -> one stat per (sample, channel)."""
    mu  = x.mean(axis=(2, 3), keepdims=True)
    var = x.var(axis=(2, 3), keepdims=True)
    x_hat = (x - mu) / np.sqrt(var + eps)
    return gamma * x_hat + beta

## Reproduce the wiki page's worked trace

$N{=}1$, $C{=}4$ channels, $H{=}1, W{=}2$ (2 spatial values per channel),
grouped into $G{=}2$ groups of 2 channels each.

In [ ]:
x_ex = np.array([[2.0, 4.0],
                  [6.0, 10.0],
                  [1.0, 3.0],
                  [0.0, 8.0]]).reshape(1, 4, 1, 2)   # (N=1, C=4, H=1, W=2)

gamma_ex = np.ones((1, 4, 1, 1))
beta_ex  = np.zeros((1, 4, 1, 1))

in_out = instance_norm(x_ex, gamma_ex, beta_ex, eps=0)
print("InstanceNorm (each channel alone):")
for c in range(4):
    print(f"  c{c}: {in_out[0, c, 0].round(3)}")

gn_out = group_norm(x_ex, G=2, gamma=gamma_ex, beta=beta_ex, eps=0)
print("\nGroupNorm, G=2 (pools channels 0-1 and 2-3):")
for c in range(4):
    print(f"  c{c}: {gn_out[0, c, 0].round(3)}")

In [ ]:
# Expected from the wiki page:
expected_instance = np.array([[-1, 1], [-1, 1], [-1, 1], [-1, 1]])
assert np.allclose(in_out[0, :, 0, :], expected_instance, atol=1e-9)

expected_group = np.array([
    [-1.183, -0.507],
    [ 0.169,  1.521],
    [-0.649,  0.000],
    [-0.974,  1.622],
])
assert np.allclose(gn_out[0, :, 0, :], expected_group, atol=1e-3)
print("Matches the wiki page's worked trace.")

## The two special cases: $G{=}1 \equiv$ LayerNorm, $G{=}C \equiv$ InstanceNorm

`group_norm` with `G=1` pools *all* channels together (LayerNorm's reduction
set), and with `G=C` puts every channel in its own group of one
(InstanceNorm's reduction set). Both identities should hold exactly, for any
tensor.

In [ ]:
N, C, H, W = 2, 8, 5, 5
x = rng.normal(0, 3, size=(N, C, H, W)) + rng.normal(0, 1, size=(N, C, 1, 1))  # per-channel offset
gamma = np.ones((1, C, 1, 1))
beta  = np.zeros((1, C, 1, 1))

gn_1 = group_norm(x, G=1, gamma=gamma, beta=beta)
ln   = layer_norm(x, gamma, beta)
assert np.allclose(gn_1, ln), "G=1 GroupNorm should equal LayerNorm"

gn_C = group_norm(x, G=C, gamma=gamma, beta=beta)
inst = instance_norm(x, gamma, beta)
assert np.allclose(gn_C, inst), "G=C GroupNorm should equal InstanceNorm"

print("Confirmed: GroupNorm(G=1) == LayerNorm, and GroupNorm(G=C) == InstanceNorm.")

### Validate: GroupNorm interpolates the whole family

GroupNorm pools statistics over $G$ channel groups per sample. At $G=1$ (all channels one
group) it equals **LayerNorm**; at $G=C$ (each channel alone) it equals **InstanceNorm**. And
any of these standardizes its slice to zero mean. We confirm the two endpoints and the
standardization.

In [ ]:
bn = batch_norm(x, gamma, beta)
assert np.allclose(bn.mean(axis=(0, 2, 3)), 0, atol=1e-5), 'normalization standardizes each channel to zero mean'
assert np.allclose(group_norm(x, G=1, gamma=gamma, beta=beta), layer_norm(x, gamma, beta)), 'GroupNorm(G=1) = LayerNorm'
assert np.allclose(group_norm(x, G=C, gamma=gamma, beta=beta), instance_norm(x, gamma, beta)), 'GroupNorm(G=C) = InstanceNorm'
print('✅ GroupNorm interpolates LayerNorm (G=1) <-> InstanceNorm (G=C) — one knob for the whole family')

## InstanceNorm removes per-instance, per-channel contrast

The callout on the wiki page claims: scaling and shifting a single channel of
a single sample ($x \mapsto a\cdot x + b$, $a>0$) leaves InstanceNorm's output
for that channel unchanged, because $\mu_{n,c}, \sigma_{n,c}$ absorb the
transform. We verify this numerically below by applying a different random
contrast/brightness shift to every (sample, channel) slice.

In [ ]:
N, C, H, W = 4, 6, 8, 8
x = rng.normal(0, 1, size=(N, C, H, W)) * rng.uniform(1, 5, size=(N, C, 1, 1))
gamma = np.ones((1, C, 1, 1))
beta  = np.zeros((1, C, 1, 1))

a = rng.uniform(0.5, 3.0, size=(N, C, 1, 1))   # per-(sample,channel) contrast
b = rng.uniform(-10, 10, size=(N, C, 1, 1))    # per-(sample,channel) brightness shift
x_transformed = a * x + b

out_original    = instance_norm(x, gamma, beta)
out_transformed = instance_norm(x_transformed, gamma, beta)

assert np.allclose(out_original, out_transformed, atol=1e-4)
print("InstanceNorm output is identical before and after the per-channel contrast/brightness shift.")
print("max abs difference:", np.abs(out_original - out_transformed).max())

### Validate: InstanceNorm is invariant to per-channel contrast & brightness

Because InstanceNorm standardizes each (sample, channel) map on its own, rescaling and shifting
a channel ($a x + b$) leaves the output **unchanged** — the affine-invariance that makes it a
staple of style transfer. We confirm.

In [ ]:
print(f'max |IN(x) - IN(a*x + b)| = {np.abs(out_original - out_transformed).max():.2e}')
assert np.allclose(out_original, out_transformed), 'InstanceNorm is invariant to per-(sample,channel) affine changes'
print('\n✅ InstanceNorm erases per-channel contrast/brightness — why it suits style transfer')

## Visualizing per-channel activation statistics under each variant

We build a toy `(N, C, H, W)` tensor with deliberately different per-channel
scales (to mimic real feature maps, where different channels fire at very
different magnitudes), then compare the per-channel mean and standard
deviation of the output under each of the four normalization variants.

In [ ]:
N, C, H, W = 8, 6, 6, 6
channel_scale = np.array([1.0, 3.0, 0.3, 5.0, 2.0, 0.5]).reshape(1, C, 1, 1)
channel_shift = np.array([0.0, 4.0, -2.0, 6.0, -3.0, 1.0]).reshape(1, C, 1, 1)
x_raw = rng.normal(0, 1, size=(N, C, H, W)) * channel_scale + channel_shift

gamma = np.ones((1, C, 1, 1))
beta  = np.zeros((1, C, 1, 1))

variants = {
    'raw (no norm)': x_raw,
    'BatchNorm':     batch_norm(x_raw, gamma, beta),
    'LayerNorm':     layer_norm(x_raw, gamma, beta),
    'GroupNorm (G=2)': group_norm(x_raw, 2, gamma, beta),
    'InstanceNorm':  instance_norm(x_raw, gamma, beta),
}

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
colors = ['#64748b', '#6366f1', '#22d3ee', '#f59e0b', '#f43f5e']
width = 0.15

for i, (name, out) in enumerate(variants.items()):
    per_channel_mean = out.mean(axis=(0, 2, 3))
    per_channel_std  = out.std(axis=(0, 2, 3))
    xs = np.arange(C) + (i - 2) * width
    axes[0].bar(xs, per_channel_mean, width=width, color=colors[i], label=name)
    axes[1].bar(xs, per_channel_std, width=width, color=colors[i], label=name)

axes[0].set_title('Per-channel activation mean'); axes[0].set_xlabel('channel')
axes[1].set_title('Per-channel activation std');  axes[1].set_xlabel('channel')
axes[0].axhline(0, color='white', lw=0.5)
axes[1].axhline(1, color='white', lw=0.5, ls='--')
axes[1].legend(loc='upper right', fontsize=8)
plt.suptitle('Raw channels have wildly different scale/shift; every normalizer pulls them toward mean 0 / std 1', y=1.03)
plt.tight_layout()
plt.show()

print("Raw per-channel std (before any normalization):", x_raw.std(axis=(0, 2, 3)).round(2))
print("BatchNorm collapses each channel to std 1 using the *whole batch* for that channel.")
print("GroupNorm/InstanceNorm do it per-sample, so they work identically at N=1.")

## Gotchas & tradeoffs

| Gotcha | Consequence |
|--------|-------------|
| **BatchNorm at small batch** | statistics are noisy / collapse (demo) — use GroupNorm |
| **choosing G** | too few/many groups changes behaviour; $G=32$ is a common default |
| **train/eval BatchNorm** | uses running stats at eval — a common bug source |
| **InstanceNorm on classification** | erasing contrast can throw away useful signal |
| **normalization + residuals** | placement (pre/post) affects training stability |

Demo: BatchNorm collapses at batch size 1; GroupNorm does not.

In [ ]:
# The reason GroupNorm/InstanceNorm exist: BATCHNORM DEPENDS ON THE BATCH. With a single sample
# and no spatial extent, each channel is one number, so BatchNorm's per-channel variance is 0 and
# the output COLLAPSES to zero. GroupNorm normalizes ACROSS channels within the sample, so it is
# batch-independent and survives. We confirm the collapse and the fix.
x1 = rng.normal(0, 2, size=(1, 8, 1, 1))
g1, b1 = np.ones((1, 8, 1, 1)), np.zeros((1, 8, 1, 1))
bn1 = batch_norm(x1, g1, b1)
gn1 = group_norm(x1, G=2, gamma=g1, beta=b1)
print(f'single-sample output std: BatchNorm={bn1.std():.3f}  GroupNorm={gn1.std():.3f}')
assert bn1.std() < 1e-3, 'BatchNorm collapses on a single sample (batch variance ~0)'
assert gn1.std() > 0.1, 'GroupNorm is batch-independent -> robust to tiny batches'
print('\nBatchNorm breaks at batch size 1 (detection, video, huge models) -> GroupNorm/LayerNorm do not.')

## ✏️ Your turn

**Task — BatchNorm's information collapse at batch size 1.** Consider a
sample *after* global average pooling, so there is no spatial dimension left:
`x` has shape `(1, C, 1, 1)` — one scalar per channel, one sample. BatchNorm's
reduction set is $(N, H, W)$; with $N{=}1, H{=}1, W{=}1$ that set has exactly
**one element per channel**, so `x.var(axis=(0,2,3))` is exactly `0` for every
channel (a single number has zero spread around its own mean). That makes
`x_hat = (x - mu) / sqrt(0 + eps)` exactly `0` for every channel too, since
`x - mu = 0` identically — the normalized output collapses to the constant
`beta`, throwing away every bit of per-channel signal in `x`.

GroupNorm doesn't have this problem: with $G < C$, each group still pools
**multiple channels** together even at $N{=}1$, so the group variance is
generally nonzero and the per-channel signal survives.

Implement `single_sample_diversity(x, G, gamma, beta, eps=1e-5)` that, for a
single sample `x` of shape `(1, C, 1, 1)`:
1. computes `bn_out = batch_norm(x, gamma, beta, eps)`,
2. computes `gn_out = group_norm(x, G, gamma, beta, eps)`,
3. returns `(bn_out.std(), gn_out.std())` — the **overall** standard
   deviation (a single scalar each, i.e. `.std()` with no `axis`) of each
   output tensor.

In [ ]:
def single_sample_diversity(x, G, gamma, beta, eps=1e-5):
    assert x.shape[0] == 1 and x.shape[2] == 1 and x.shape[3] == 1, \
        "expects a single sample with no spatial dimension, shape (1, C, 1, 1)"
    # TODO(you): compute bn_out = batch_norm(x, gamma, beta, eps)
    # TODO(you): compute gn_out = group_norm(x, G, gamma, beta, eps)
    # TODO(you): return (bn_out.std(), gn_out.std())
    return ...


C = 8
x_single = rng.normal(0, 2, size=(1, C, 1, 1))
gamma = np.ones((1, C, 1, 1))
beta  = np.zeros((1, C, 1, 1))

result = single_sample_diversity(x_single, G=4, gamma=gamma, beta=beta, eps=1e-5)
if result is not None:
    bn_std, gn_std = result
    print(f"BatchNorm output std at N=1, no spatial dims:  {bn_std:.6f}")
    print(f"GroupNorm output std at N=1, no spatial dims:  {gn_std:.6f}")
    assert bn_std < 1e-6, "BatchNorm should collapse to the constant beta (std ~ 0) at N=1"
    assert gn_std > 0.5, "GroupNorm should retain meaningful per-channel signal at N=1"
    print("Confirmed: BatchNorm collapses all signal to a constant at N=1; GroupNorm does not.")

<details>
<summary>Solution</summary>

```python
def single_sample_diversity(x, G, gamma, beta, eps=1e-5):
    assert x.shape[0] == 1 and x.shape[2] == 1 and x.shape[3] == 1, \
        "expects a single sample with no spatial dimension, shape (1, C, 1, 1)"
    bn_out = batch_norm(x, gamma, beta, eps)
    gn_out = group_norm(x, G, gamma, beta, eps)
    return bn_out.std(), gn_out.std()
```

With `N=1` and no spatial dimensions, BatchNorm's reduction set `(N,H,W)` has
exactly one element per channel, so `x - mu` is identically `0` and every
channel normalizes to `beta` — a constant, independent of what `x` actually
was. GroupNorm's reduction set `(C/G, H, W)` still contains `C/G = 2` distinct
channel values even at `N=1`, so its per-group mean/variance are generally
nonzero and the two channels in a group normalize to two *different* values
(recall from the worked trace: a 2-element group always normalizes to
$\pm 1$) — the per-channel signal survives instead of collapsing.
</details>

## Key takeaways

- **The norms differ only in reduction axes:** batch / sample / channel-group / channel.
- **GroupNorm interpolates:** $G=1$ = LayerNorm, $G=C$ = InstanceNorm (verified).
- **InstanceNorm is affine-invariant** per channel (verified) — great for style transfer.
- **GroupNorm is batch-independent:** it survives batch size 1 where BatchNorm collapses (demo).